Latihan mandiri
Gunakan soup dari HTML lokal untuk menjawab pertanyaan berikut.

1. Ambil hanya judul buku yang statusnya Tersedia.
2. Hitung rata-rata harga buku.
3. Ambil semua atribut data-id dengan satu CSS selector.
4. Ubah fungsi ekstraksi agar stok yang hilang menjadi string Tidak diketahui.

Coba kerjakan sebelum membuka solusi.

### Latihan Mandiri - Web Scraping dengan Beautiful Soup

In [13]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from bs4.element import Tag
import pandas as pd

In [14]:
HTML_CONTOH = """
<!doctype html>
<html lang="id">
  <head><title>Toko Buku Data</title></head>
  <body>
    <h1>Buku Pilihan</h1>
    <section id="katalog">
      <article class="buku unggulan" data-id="B001">
        <h2 class="judul">Dasar Data Mining</h2>
        <p class="harga">Rp125.000</p>
        <p class="stok">Tersedia</p>
        <a href="/buku/dasar-data-mining">Detail</a>
      </article>
      <article class="buku" data-id="B002">
        <h2 class="judul">Python untuk Analisis Data</h2>
        <p class="harga">Rp149.500</p>
        <p class="stok habis">Habis</p>
        <a href="/buku/python-analisis">Detail</a>
      </article>
      <article class="buku" data-id="B003">
        <h2 class="judul">Statistika Praktis</h2>
        <p class="harga">Rp98.000</p>
        <!-- Elemen stok sengaja tidak tersedia -->
        <a href="/buku/statistika-praktis">Detail</a>
      </article>
    </section>
  </body>
</html>
"""

BASE_URL_CONTOH = "https://contoh.invalid"
soup = BeautifulSoup(HTML_CONTOH, "html.parser")

In [15]:
def teks_atau_none(induk: Tag, selector: str) -> str | None:
    elemen = induk.select_one(selector)
    return elemen.get_text(" ", strip=True) if elemen else None


def harga_ke_int(teks_harga: str | None) -> int | None:
    if teks_harga is None:
        return None
    digit = "".join(karakter for karakter in teks_harga if karakter.isdigit())
    return int(digit) if digit else None


def ekstrak_buku(dokumen: BeautifulSoup) -> tuple[dict[str, object], ...]:
    def ekstrak_satu(kartu: Tag) -> dict[str, object]:
        tautan = kartu.select_one("a[href]")
        href = tautan.get("href") if tautan else None
        return {
            "id": kartu.get("data-id"),
            "judul": teks_atau_none(kartu, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(kartu, ".harga")),
            "stok": teks_atau_none(kartu, ".stok"),
            "url": urljoin(BASE_URL_CONTOH, str(href)) if href else None,
        }

    return tuple(ekstrak_satu(kartu) for kartu in dokumen.select("article.buku"))


data_buku = ekstrak_buku(soup)
df_buku = pd.DataFrame(data_buku)
df_buku

,id,judul,harga_rupiah,stok,url
0,B001,Dasar Data Mining,125000,Tersedia,https://contoh.invalid/buku/dasar-data-mining
1,B002,Python untuk Analisis Data,149500,Habis,https://contoh.invalid/buku/python-analisis
2,B003,Statistika Praktis,98000,NaN,https://contoh.invalid/buku/statistika-praktis


##### Soal 1: Judul buku yang statusnya Tersedia

In [16]:
judul_tersedia = [
    kartu.select_one(".judul").get_text(strip=True)
    for kartu in soup.select("article.buku")
    if kartu.select_one(".stok") and kartu.select_one(".stok").get_text(strip=True) == "Tersedia"
]
judul_tersedia

['Dasar Data Mining']

##### Soal 2: Rata-rata harga buku

In [17]:
rata_rata_harga = df_buku["harga_rupiah"].mean()
rata_rata_harga

np.float64(124166.66666666667)

##### Soal 3: Semua atribut data-id dengan satu CSS selector

In [18]:
semua_id = [tag.get("data-id") for tag in soup.select("article.buku")]
semua_id

['B001', 'B002', 'B003']

##### Soal 4: Fungsi ekstraksi dengan stok "Tidak diketahui"

In [19]:
def ekstrak_buku_v2(dokumen: BeautifulSoup) -> tuple[dict[str, object], ...]:
    def ekstrak_satu(kartu: Tag) -> dict[str, object]:
        tautan = kartu.select_one("a[href]")
        href = tautan.get("href") if tautan else None
        return {
            "id": kartu.get("data-id"),
            "judul": teks_atau_none(kartu, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(kartu, ".harga")),
            "stok": teks_atau_none(kartu, ".stok") or "Tidak diketahui",
            "url": urljoin(BASE_URL_CONTOH, str(href)) if href else None,
        }

    return tuple(ekstrak_satu(kartu) for kartu in dokumen.select("article.buku"))


df_buku_v2 = pd.DataFrame(ekstrak_buku_v2(soup))
df_buku_v2

,id,judul,harga_rupiah,stok,url
0,B001,Dasar Data Mining,125000,Tersedia,https://contoh.invalid/buku/dasar-data-mining
1,B002,Python untuk Analisis Data,149500,Habis,https://contoh.invalid/buku/python-analisis
2,B003,Statistika Praktis,98000,Tidak diketahui,https://contoh.invalid/buku/statistika-praktis
